# Step 2: Train Diff-SAE

This notebook demonstrates training a BatchTopK SAE on activation differences.

In [ ]:
import sys
sys.path.append('..')

import torch
import yaml
from src.models import BatchTopKSAE
from src.training import SAETrainer
from src.utils import DiffActivationCollector, prepare_training_data

## Load Configuration

In [ ]:
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("SAE Configuration:")
print(f"  d_model: {config['sae']['d_model']}")
print(f"  n_latents: {config['sae']['n_latents']}")
print(f"  k (top-k): {config['sae']['k']}")
print(f"  Learning rate: {config['sae']['learning_rate']}")

## Load Activations

In [ ]:
activations_path = '../data/activations/diff_activations.pkl'

activations = DiffActivationCollector.load_activations(activations_path)

print(f"\nLoaded activations:")
for key, value in activations.items():
    print(f"  {key}: {value.shape}")

## Prepare Training Data

In [ ]:
# Flatten sequence dimension for training
train_data = prepare_training_data(
    activations['diff'],
    flatten_sequence=True
)

print(f"Training data shape: {train_data.shape}")
print(f"Number of training samples: {len(train_data):,}")

## Initialize SAE

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

sae = BatchTopKSAE(
    d_model=config['sae']['d_model'],
    n_latents=config['sae']['n_latents'],
    k=config['sae']['k']
)

print(f"\nSAE initialized:")
print(f"  Parameters: {sum(p.numel() for p in sae.parameters()):,}")
print(sae.get_config())

## Test Forward Pass

In [ ]:
# Test with a small batch
test_batch = train_data[:4].to(device)
sae = sae.to(device)

with torch.no_grad():
    output = sae(test_batch, return_components=True)

print("Test forward pass:")
print(f"  Input shape: {test_batch.shape}")
print(f"  Reconstruction shape: {output['reconstruction'].shape}")
print(f"  Latents shape: {output['latents'].shape}")
print(f"  L0 (active features): {output['l0']:.2f}")

# Compute reconstruction error
mse = torch.mean((output['reconstruction'] - test_batch) ** 2)
print(f"  Initial MSE: {mse:.6f}")

## Train SAE

In [ ]:
# Create trainer
trainer = SAETrainer(sae, config, device=device)

print("Starting training...")
print(f"Num epochs: {config['sae']['num_epochs']}")
print(f"Batch size: {config['sae']['batch_size']}")

In [ ]:
# Train the SAE
trainer.train(train_data, num_epochs=config['sae']['num_epochs'])

## Evaluate Trained SAE

In [ ]:
# Test on a validation batch
val_batch = train_data[-100:].to(device)

sae.eval()
with torch.no_grad():
    output = sae(val_batch, return_components=True)
    loss_dict = sae.compute_loss(val_batch)

print("\nValidation Results:")
print(f"  MSE Loss: {loss_dict['mse_loss']:.6f}")
print(f"  L0 (active features): {loss_dict['l0']:.2f}")

# Check sparsity
sparsity = (output['latents'] == 0).float().mean()
print(f"  Sparsity: {sparsity:.4f} ({sparsity * 100:.2f}% zeros)")

## Analyze Features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get feature magnitudes
with torch.no_grad():
    feature_mags = sae.get_feature_magnitudes(train_data[:1000].to(device))

# Plot distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(feature_mags.cpu().numpy(), bins=50)
plt.xlabel('Feature Magnitude')
plt.ylabel('Count')
plt.title('Distribution of Feature Activations')
plt.yscale('log')

plt.subplot(1, 2, 2)
sorted_mags, _ = torch.sort(feature_mags, descending=True)
plt.plot(sorted_mags.cpu().numpy())
plt.xlabel('Feature Rank')
plt.ylabel('Magnitude')
plt.title('Feature Magnitudes (Sorted)')
plt.yscale('log')

plt.tight_layout()
plt.show()

# Find dead features
dead_features = (feature_mags == 0).sum().item()
print(f"\nDead features (never activated): {dead_features} / {config['sae']['n_latents']}")